# OptGPDCNN-inspired implementation — Cardiovascular Disease (70,000 records)

**Dataset:** `cardio_train.csv` (Kaggle Cardiovascular Disease dataset)  
**Task:** Binary classification (`cardio`: 0 = no cardiovascular disease, 1 = disease)  
**Framework:** PyTorch + scikit-learn

This notebook adapts the earlier Cleveland implementation to the 70,000-row `cardio_train.csv` dataset.

Pipeline:
1. Load and validate `cardio_train.csv`
2. Clean obvious invalid measurements and prepare the target
3. Stratified train/validation/test split
4. Imputation, categorical encoding, and scaling
5. SPE-ACGAN-inspired training-set-only balancing (automatically skipped when classes are already balanced)
6. Improved CapsNet-inspired feature extraction
7. EAFT-inspired feature selection
8. GPDCNN-inspired classifier
9. MCSA-inspired hyperparameter search
10. Final held-out test evaluation and artifact saving

## Reproducibility note

This is an **educational, reproduction-oriented adaptation**, not the authors' exact source code. Several paper components (SPE-ACGAN progressive/equilibrium details, full Improved CapsNet specification, EAFT, and MCSA) are not sufficiently specified to reproduce uniquely. The implementations below approximate the described ideas and adapt the network to tabular data.

The 70,000-row dataset is not the Cleveland dataset. Its results must not be presented as a direct reproduction or direct comparison with Cleveland results from the paper.

**Important:** the Kaggle dataset is already almost perfectly class-balanced (roughly half 0 and half 1). Therefore, the notebook skips synthetic oversampling by default when the training-class imbalance is small. Generating artificial records when they are not needed can make performance worse.

## 0. Install dependencies

Run once if needed. Restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib seaborn torch joblib

## 1. Imports, seed, and device

In [ ]:
import os, random, copy, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    balanced_accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Load `cardio_train.csv`

Put `cardio_train.csv` in the **same working directory** as this notebook. The file is commonly semicolon-separated (`;`), not comma-separated.

If Jupyter cannot find it, inspect `os.getcwd()` and either move the CSV into that folder or set `DATA_FILE` to its full path.

In [ ]:
print("Notebook working directory:", os.getcwd())
print("Files here (first 30):", os.listdir()[:30])

DATA_FILE = "cardio_train.csv"  # Or use a full path, e.g. r"C:\\Users\\you\\Downloads\\cardio_train.csv"

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"Cannot find {DATA_FILE!r} in {os.getcwd()}. "
        "Place cardio_train.csv in this folder or set DATA_FILE to its full path."
    )

# The original Kaggle CSV uses semicolons.
raw = pd.read_csv(DATA_FILE, sep=";")

# Fallback for comma-separated variants.
if raw.shape[1] == 1:
    raw = pd.read_csv(DATA_FILE)

raw.columns = [c.strip().lower() for c in raw.columns]
print("Loaded shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())

## 3. Validate columns, clean invalid records, and prepare target

Expected columns: `id, age, gender, height, weight, ap_hi, ap_lo, cholesterol, gluc, smoke, alco, active, cardio`.

- `age` is recorded in **days**, so it is converted to years.
- `id` is an identifier and is removed.
- Clearly implausible measurement rows are removed using broad sanity bounds. The number removed is printed.
- Target `cardio` is retained as `target` (0/1).

These cleaning limits are explicit choices for this experiment, not a claim that they are the only medically valid thresholds.

In [ ]:
required = {
    "age", "gender", "height", "weight", "ap_hi", "ap_lo",
    "cholesterol", "gluc", "smoke", "alco", "active", "cardio"
}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

data = raw.copy()

# Ensure numeric columns are numeric; malformed values become missing.
for col in required:
    data[col] = pd.to_numeric(data[col], errors="coerce")

before = len(data)

# Remove missing target rows and rows with invalid target labels.
data = data.dropna(subset=["cardio"])
data = data[data["cardio"].isin([0, 1])].copy()

# Convert age from days to years.
data["age"] = data["age"] / 365.25

# Broad measurement sanity filters. Adjust only if you have a documented reason.
valid = (
    data["age"].between(18, 100) &
    data["height"].between(120, 220) &
    data["weight"].between(30, 200) &
    data["ap_hi"].between(60, 240) &
    data["ap_lo"].between(40, 160) &
    (data["ap_hi"] > data["ap_lo"])
)
data = data.loc[valid].copy()

# Remove identifier; it has no clinical predictive meaning.
if "id" in data.columns:
    data = data.drop(columns=["id"])

data = data.rename(columns={"cardio": "target"})
data["target"] = data["target"].astype(int)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {len(data):,}")
print(f"Rows removed:         {before - len(data):,}")
print("\nTarget distribution:")
display(data["target"].value_counts().sort_index().rename(index={0:"No disease", 1:"Disease"}))
display((100 * data["target"].value_counts(normalize=True)).sort_index().rename(index={0:"No disease %", 1:"Disease %"}))
display(data.describe().T)

## 4. Train / validation / test split

Use a stratified 70% / 15% / 15% split. The test set is held out and must not be used for preprocessing fit, feature selection, or hyperparameter search.

In [ ]:
numeric_cols = ["age", "height", "weight", "ap_hi", "ap_lo"]
categorical_cols = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

X_raw = data[numeric_cols + categorical_cols].copy()
y = data["target"].to_numpy(dtype=np.int64)

X_trainval_raw, X_test_raw, y_trainval, y_test = train_test_split(
    X_raw, y, test_size=0.15, random_state=SEED, stratify=y
)
# 15% of total for validation: 0.15 / 0.85 of the remaining data.
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_trainval_raw, y_trainval, test_size=(0.15 / 0.85),
    random_state=SEED, stratify=y_trainval
)

print("Train:", X_train_raw.shape, "Validation:", X_val_raw.shape, "Test:", X_test_raw.shape)
for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    counts = np.bincount(labels, minlength=2)
    print(f"{name:10s} No disease={counts[0]:,}, Disease={counts[1]:,}, disease rate={labels.mean():.3f}")

## 5. Preprocessing

Numeric variables receive median imputation and Min-Max scaling. Categorical/binary variables receive most-frequent imputation and one-hot encoding.

The preprocessor is fitted on the training partition only.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

X_train = preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_val = preprocessor.transform(X_val_raw).astype(np.float32)
X_test = preprocessor.transform(X_test_raw).astype(np.float32)

feature_names = preprocessor.get_feature_names_out()
print("Encoded feature count:", X_train.shape[1])
print("Train/validation/test matrices:", X_train.shape, X_val.shape, X_test.shape)

## 6. Stage 2 — SPE-ACGAN-inspired class balancing

The GAN is trained only on the training partition. Since this dataset is nearly balanced, oversampling is normally unnecessary. By default, the code skips GAN training if the minority/majority ratio is at least 0.95.

If the dataset becomes meaningfully imbalanced after cleaning, set `FORCE_GAN = True` or lower `BALANCE_THRESHOLD`. The GAN implementation is an ACGAN-inspired approximation, not an exact SPE-ACGAN reproduction.

In [ ]:
class TabularGenerator(nn.Module):
    def __init__(self, noise_dim, n_classes, n_features, emb_dim=8):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(noise_dim + emb_dim, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Linear(256, n_features), nn.Sigmoid()
        )
    def forward(self, z, labels):
        return self.net(torch.cat([z, self.label_emb(labels)], dim=1))

class TabularDiscriminator(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(n_features, 256), nn.LeakyReLU(0.2), nn.Dropout(0.15),
            nn.Linear(256, 128), nn.LeakyReLU(0.2)
        )
        self.validity = nn.Linear(128, 1)
        self.classifier = nn.Linear(128, n_classes)
    def forward(self, x):
        h = self.shared(x)
        return self.validity(h), self.classifier(h)

def train_acgan(X, y, epochs=25, batch_size=512, noise_dim=32, lr=2e-4):
    n_features = X.shape[1]
    n_classes = 2
    G = TabularGenerator(noise_dim, n_classes, n_features).to(DEVICE)
    D = TabularDiscriminator(n_features, n_classes).to(DEVICE)
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    adv_loss = nn.BCEWithLogitsLoss()
    cls_loss = nn.CrossEntropyLoss()

    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.long)),
        batch_size=batch_size, shuffle=True, drop_last=True
    )
    for epoch in range(epochs):
        G.train(); D.train()
        for real_x, real_y in loader:
            real_x, real_y = real_x.to(DEVICE), real_y.to(DEVICE)
            bs = len(real_x)
            real_valid = torch.ones(bs, 1, device=DEVICE)
            fake_valid = torch.zeros(bs, 1, device=DEVICE)

            # Train discriminator.
            z = torch.randn(bs, noise_dim, device=DEVICE)
            sampled_labels = torch.randint(0, n_classes, (bs,), device=DEVICE)
            fake_x = G(z, sampled_labels).detach()
            real_logits, real_cls = D(real_x)
            fake_logits, _ = D(fake_x)
            d_loss = (
                adv_loss(real_logits, real_valid) +
                adv_loss(fake_logits, fake_valid) +
                cls_loss(real_cls, real_y)
            )
            opt_d.zero_grad(); d_loss.backward(); opt_d.step()

            # Train generator.
            z = torch.randn(bs, noise_dim, device=DEVICE)
            gen_labels = torch.randint(0, n_classes, (bs,), device=DEVICE)
            gen_x = G(z, gen_labels)
            gen_logits, gen_cls = D(gen_x)
            g_loss = adv_loss(gen_logits, real_valid) + cls_loss(gen_cls, gen_labels)
            opt_g.zero_grad(); g_loss.backward(); opt_g.step()

        if (epoch + 1) % 5 == 0:
            print(f"ACGAN epoch {epoch+1:3d}/{epochs}, D={d_loss.item():.4f}, G={g_loss.item():.4f}")
    return G

def balance_with_acgan(X, y, threshold=0.95, force=False, gan_epochs=25):
    counts = np.bincount(y, minlength=2)
    ratio = counts.min() / max(counts.max(), 1)
    print("Training class counts:", counts, "minority/majority ratio:", round(ratio, 4))
    if ratio >= threshold and not force:
        print("Classes are already sufficiently balanced. Skipping GAN augmentation.")
        return X.copy(), y.copy(), None

    minority_class = int(np.argmin(counts))
    n_to_generate = int(counts.max() - counts.min())
    if n_to_generate <= 0:
        print("No augmentation required.")
        return X.copy(), y.copy(), None

    G = train_acgan(X, y, epochs=gan_epochs)
    G.eval()
    generated = []
    with torch.no_grad():
        for start in range(0, n_to_generate, 1024):
            n = min(1024, n_to_generate - start)
            z = torch.randn(n, 32, device=DEVICE)
            labels = torch.full((n,), minority_class, dtype=torch.long, device=DEVICE)
            generated.append(G(z, labels).cpu().numpy())
    X_syn = np.vstack(generated).astype(np.float32)
    y_syn = np.full(n_to_generate, minority_class, dtype=np.int64)
    return np.vstack([X, X_syn]), np.concatenate([y, y_syn]), G

FORCE_GAN = False
BALANCE_THRESHOLD = 0.95
X_train_bal, y_train_bal, gan = balance_with_acgan(
    X_train, y_train, threshold=BALANCE_THRESHOLD, force=FORCE_GAN, gan_epochs=25
)
print("After Stage 2:", X_train_bal.shape, np.bincount(y_train_bal, minlength=2))

### Class distribution before and after Stage 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(["No disease", "Disease"], np.bincount(y_train, minlength=2))
axes[0].set_title("Training data before Stage 2")
axes[1].bar(["No disease", "Disease"], np.bincount(y_train_bal, minlength=2))
axes[1].set_title("Training data after Stage 2")
for ax in axes:
    ax.set_ylabel("Number of records")
plt.tight_layout()
plt.show()

## 7. Stage 3 — Improved CapsNet-inspired feature extraction

The tabular feature vector is treated as a 1-D sequence. The model creates primary capsules, applies dynamic routing to feature capsules, and reconstructs the input from capsule features. The training objective combines a class margin loss with a reconstruction penalty (`beta = 0.0005`).

This is a practical tabular adaptation of the paper's description.

In [ ]:
def squash(v, dim=-1, eps=1e-8):
    norm_sq = (v * v).sum(dim=dim, keepdim=True)
    return (norm_sq / (1.0 + norm_sq)) * v / torch.sqrt(norm_sq + eps)

class DynamicRouting(nn.Module):
    def __init__(self, n_primary=16, n_feature_caps=12, in_dim=8, out_dim=8, routing_iters=3):
        super().__init__()
        self.n_primary = n_primary
        self.n_feature_caps = n_feature_caps
        self.out_dim = out_dim
        self.routing_iters = routing_iters
        self.W = nn.Parameter(0.01 * torch.randn(1, n_primary, n_feature_caps, out_dim, in_dim))
    def forward(self, x):
        # x: [batch, primary_capsules, in_dim]
        u_hat = torch.einsum("bpi,apfoi->bpfo", x, self.W)
        logits = torch.zeros(x.size(0), self.n_primary, self.n_feature_caps, device=x.device)
        for r in range(self.routing_iters):
            coupling = torch.softmax(logits, dim=2)
            s = (coupling.unsqueeze(-1) * u_hat).sum(dim=1)
            v = squash(s, dim=-1)
            if r < self.routing_iters - 1:
                agreement = (u_hat * v.unsqueeze(1)).sum(dim=-1)
                logits = logits + agreement
        return v

class ImprovedCapsNetTabular(nn.Module):
    def __init__(self, n_features, n_primary=16, primary_dim=8, n_feature_caps=12, cap_dim=8):
        super().__init__()
        self.n_features = n_features
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, n_primary * primary_dim, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.n_primary = n_primary
        self.primary_dim = primary_dim
        self.routing = DynamicRouting(n_primary, n_feature_caps, primary_dim, cap_dim)
        self.decoder = nn.Sequential(
            nn.Linear(n_feature_caps * cap_dim, 128), nn.ReLU(),
            nn.Linear(128, n_features), nn.Sigmoid()
        )
        self.classifier = nn.Linear(n_feature_caps * cap_dim, 2)
    def forward(self, x):
        h = self.conv(x.unsqueeze(1))
        # Adaptive pooling guarantees a fixed number of primary capsules.
        h = F.adaptive_avg_pool1d(h, self.n_primary)
        h = h.transpose(1, 2).contiguous()
        h = h.view(x.size(0), self.n_primary, self.primary_dim)
        caps = self.routing(h)
        flat = caps.flatten(1)
        logits = self.classifier(flat)
        reconstruction = self.decoder(flat)
        return logits, flat, reconstruction

def capsule_margin_loss(logits, labels):
    probs = torch.softmax(logits, dim=1)
    onehot = F.one_hot(labels, num_classes=2).float()
    pos = onehot * F.relu(0.9 - probs).pow(2)
    neg = (1 - onehot) * F.relu(probs - 0.1).pow(2)
    return (pos + 0.5 * neg).sum(dim=1).mean()

def train_capsnet(X, y, n_features, epochs=20, batch_size=512, beta=0.0005):
    model = ImprovedCapsNetTabular(n_features).to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.long)),
        batch_size=batch_size, shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits, _, recon = model(xb)
            loss = capsule_margin_loss(logits, yb) + beta * F.mse_loss(recon, xb)
            optimizer.zero_grad(); loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if (epoch + 1) % 5 == 0:
            print(f"CapsNet epoch {epoch+1:3d}/{epochs}, loss={np.mean(losses):.5f}")
    return model

capsnet = train_capsnet(
    X_train_bal, y_train_bal, n_features=X_train.shape[1],
    epochs=20, batch_size=512
)

def extract_capsule_features(model, X, batch_size=1024):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.tensor(X[start:start+batch_size], dtype=torch.float32, device=DEVICE)
            _, feats, _ = model(xb)
            outputs.append(feats.cpu().numpy())
    return np.vstack(outputs).astype(np.float32)

F_train = extract_capsule_features(capsnet, X_train_bal)
F_val = extract_capsule_features(capsnet, X_val)
F_test = extract_capsule_features(capsnet, X_test)
print("Capsule feature matrices:", F_train.shape, F_val.shape, F_test.shape)

## 8. Stage 4 — EAFT-inspired feature selection

Each candidate is a binary feature mask. Fitness is validation F1 minus a small penalty for selecting too many features. To keep runtime reasonable on 70k records, fitness is evaluated on a fixed, stratified subset of the training data. Validation data is used for scoring; test data is never used.

This is an AFT-inspired wrapper approximation, not the exact EAFT algorithm.

In [ ]:
# Fixed stratified subset for faster wrapper search.
rng = np.random.default_rng(SEED)
MAX_FS_TRAIN = 12000
if len(F_train) > MAX_FS_TRAIN:
    idx_parts = []
    for cls in np.unique(y_train_bal):
        cls_idx = np.where(y_train_bal == cls)[0]
        take = min(len(cls_idx), MAX_FS_TRAIN // len(np.unique(y_train_bal)))
        idx_parts.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    fs_idx = np.array(idx_parts)
else:
    fs_idx = np.arange(len(F_train))

F_fs, y_fs = F_train[fs_idx], y_train_bal[fs_idx]

def subset_fitness(mask, Xtr, ytr, Xv, yv):
    if mask.sum() == 0:
        return 0.0
    clf = LogisticRegression(max_iter=300, class_weight="balanced", solver="liblinear", random_state=SEED)
    clf.fit(Xtr[:, mask], ytr)
    pred = clf.predict(Xv[:, mask])
    score = f1_score(yv, pred, zero_division=0)
    penalty = 0.01 * (mask.sum() / len(mask))
    return score - penalty

def eaft_inspired_select(Xtr, ytr, Xv, yv, n_agents=8, iterations=8):
    n_features = Xtr.shape[1]
    rng = np.random.default_rng(SEED)
    population = rng.integers(0, 2, size=(n_agents, n_features)).astype(int)
    for row in population:
        if row.sum() == 0:
            row[rng.integers(n_features)] = 1
    scores = np.array([subset_fitness(m.astype(bool), Xtr, ytr, Xv, yv) for m in population])
    best_i = int(np.argmax(scores))
    best_mask, best_score = population[best_i].copy(), float(scores[best_i])
    history = [best_score]
    for t in range(iterations):
        flip_p = 0.20 * (1 - (t / max(iterations, 1))) + 0.02
        for i in range(n_agents):
            candidate = population[i].copy()
            toward_best = rng.random(n_features) < (0.25 + 0.5 * (1 - flip_p))
            candidate[toward_best] = best_mask[toward_best]
            flips = rng.random(n_features) < flip_p
            candidate[flips] = 1 - candidate[flips]
            if candidate.sum() == 0:
                candidate[rng.integers(n_features)] = 1
            score = subset_fitness(candidate.astype(bool), Xtr, ytr, Xv, yv)
            if score >= scores[i]:
                population[i], scores[i] = candidate, score
            if score > best_score:
                best_mask, best_score = candidate.copy(), float(score)
        history.append(best_score)
        print(f"EAFT-inspired iteration {t+1}/{iterations}, best fitness={best_score:.4f}")
    return best_mask.astype(bool), history

selected_mask, fs_history = eaft_inspired_select(F_fs, y_fs, F_val, y_val)
if selected_mask.sum() == 0:
    selected_mask[:] = True

F_train_sel = F_train[:, selected_mask]
F_val_sel = F_val[:, selected_mask]
F_test_sel = F_test[:, selected_mask]
print("Selected features:", int(selected_mask.sum()), "/", len(selected_mask))

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(fs_history, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Best validation fitness")
plt.title("EAFT-inspired feature-selection search")
plt.grid(True)
plt.show()

## 9. Stage 5 — GPDCNN-inspired classifier

A compact 1-D CNN uses dilated convolutions and parallel multi-scale branches, followed by global average pooling and a two-class output. The tabular adaptation is intentionally smaller than a typical image CNN.

In [ ]:
class InceptionDilatedBlock1D(nn.Module):
    def __init__(self, in_ch, branch_ch=32):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv1d(in_ch, branch_ch, 1), nn.ReLU())
        self.b2 = nn.Sequential(nn.Conv1d(in_ch, branch_ch, 3, padding=1, dilation=1), nn.ReLU())
        self.b3 = nn.Sequential(nn.Conv1d(in_ch, branch_ch, 3, padding=2, dilation=2), nn.ReLU())
        self.b4 = nn.Sequential(nn.Conv1d(in_ch, branch_ch, 3, padding=4, dilation=4), nn.ReLU())
        self.project = nn.Sequential(nn.Conv1d(branch_ch * 4, branch_ch * 2, 1), nn.ReLU())
    def forward(self, x):
        return self.project(torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1))

class GPDCNN(nn.Module):
    def __init__(self, n_features, width=32, dropout=0.30):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(1, width, 3, padding=2, dilation=2), nn.ReLU())
        self.inception = InceptionDilatedBlock1D(width, width)
        self.dilated = nn.Sequential(
            nn.Conv1d(width * 2, width * 2, 3, padding=4, dilation=4),
            nn.ReLU()
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(width * 2, width), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(width, 2)
        )
    def forward(self, x):
        x = self.stem(x.unsqueeze(1))
        x = self.inception(x)
        x = self.dilated(x)
        return self.head(self.gap(x))

## 10. Stage 6 — MCSA-inspired hyperparameter search

A compact population search selects learning rate, channel width, dropout, and batch size using validation F1. Search is deliberately limited for runtime. The test partition is not used.

In [ ]:
def predict_probabilities(model, X, batch_size=2048):
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.tensor(X[start:start+batch_size], dtype=torch.float32, device=DEVICE)
            probs.append(torch.softmax(model(xb), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

def evaluate_model(model, X, y):
    probs = predict_probabilities(model, X)
    pred = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "auc": roc_auc_score(y, probs)
    }

def train_gp_dcnn(Xtr, ytr, Xv, yv, config, epochs=18, verbose=False):
    model = GPDCNN(
        n_features=Xtr.shape[1],
        width=int(config["width"]),
        dropout=float(config["dropout"])
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=float(config["lr"]), weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(
        TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                      torch.tensor(ytr, dtype=torch.long)),
        batch_size=int(config["batch_size"]), shuffle=True
    )
    best_state, best_f1 = copy.deepcopy(model.state_dict()), -1
    patience, stale = 4, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = criterion(model(xb), yb)
            optimizer.zero_grad(); loss.backward()
            optimizer.step()
        metrics = evaluate_model(model, Xv, yv)
        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break
    model.load_state_dict(best_state)
    if verbose:
        print("Best validation F1:", round(best_f1, 4))
    return model, best_f1

# Small, reproducible search space. Increase candidates/epochs only if runtime allows.
search_space = [
    {"lr": 1e-3, "width": 24, "dropout": 0.25, "batch_size": 512},
    {"lr": 5e-4, "width": 32, "dropout": 0.30, "batch_size": 512},
    {"lr": 3e-4, "width": 48, "dropout": 0.35, "batch_size": 1024},
    {"lr": 8e-4, "width": 32, "dropout": 0.20, "batch_size": 1024},
]

search_results = []
best_config, best_val_f1 = None, -1
for i, config in enumerate(search_space, 1):
    print(f"\nCandidate {i}/{len(search_space)}:", config)
    candidate_model, score = train_gp_dcnn(
        F_train_sel, y_train_bal, F_val_sel, y_val,
        config, epochs=18, verbose=False
    )
    search_results.append({**config, "validation_f1": score})
    if score > best_val_f1:
        best_val_f1, best_config = score, config.copy()

search_results_df = pd.DataFrame(search_results).sort_values("validation_f1", ascending=False)
display(search_results_df)
print("Best config:", best_config)
print("Best validation F1:", best_val_f1)

## 11. Final training and one-time held-out test evaluation

After choosing the configuration, train a fresh final classifier on the training plus validation partitions. The held-out test set is used only for the final evaluation.

The feature extractor was trained on the training partition; its validation/test outputs are produced without fitting on those partitions. For a fully nested evaluation, the capsule extractor would also need retraining on the combined train+validation partition after model selection.

In [ ]:
# Combine train and validation capsule representations for final classifier fit.
F_final = np.vstack([F_train_sel, F_val_sel]).astype(np.float32)
y_final = np.concatenate([y_train_bal, y_val]).astype(np.int64)

# Train using a small internal holdout for early stopping.
X_fit, X_es, y_fit, y_es = train_test_split(
    F_final, y_final, test_size=0.10, random_state=SEED, stratify=y_final
)
final_model, final_val_f1 = train_gp_dcnn(
    X_fit, y_fit, X_es, y_es, best_config, epochs=25, verbose=True
)

test_probs = predict_probabilities(final_model, F_test_sel)
test_pred = (test_probs >= 0.5).astype(int)

test_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "auc": roc_auc_score(y_test, test_probs)
}

print("HELD-OUT TEST METRICS")
for key, value in test_metrics.items():
    print(f"{key:>18}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test, test_pred, target_names=["No disease", "Disease"], zero_division=0
))

cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No disease", "Disease"],
            yticklabels=["No disease", "Disease"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Held-out test confusion matrix")
plt.show()

print("Predicted class counts:", np.bincount(test_pred, minlength=2))

## 12. Baseline comparisons

Compare the deep-learning pipeline against conventional models on the same preprocessed data and same held-out test partition. These are important sanity checks for a tabular dataset.

In [ ]:
baseline_rows = []

# Logistic regression
lr_model = LogisticRegression(max_iter=1500, class_weight="balanced", solver="liblinear", random_state=SEED)
lr_model.fit(X_train, y_train)
lr_prob = lr_model.predict_proba(X_test)[:, 1]
lr_pred = (lr_prob >= 0.5).astype(int)
baseline_rows.append({
    "model": "Logistic Regression",
    "accuracy": accuracy_score(y_test, lr_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, lr_pred),
    "precision": precision_score(y_test, lr_pred, zero_division=0),
    "recall": recall_score(y_test, lr_pred, zero_division=0),
    "f1": f1_score(y_test, lr_pred, zero_division=0),
    "auc": roc_auc_score(y_test, lr_prob)
})

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=3, max_features="sqrt",
    class_weight="balanced_subsample", n_jobs=-1, random_state=SEED
)
rf_model.fit(X_train, y_train)
rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)
baseline_rows.append({
    "model": "Random Forest",
    "accuracy": accuracy_score(y_test, rf_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, rf_pred),
    "precision": precision_score(y_test, rf_pred, zero_division=0),
    "recall": recall_score(y_test, rf_pred, zero_division=0),
    "f1": f1_score(y_test, rf_pred, zero_division=0),
    "auc": roc_auc_score(y_test, rf_prob)
})

baseline_rows.append({"model": "OptGPDCNN-inspired", **test_metrics})
baseline_table = pd.DataFrame(baseline_rows).set_index("model")
display(baseline_table.round(4))

## 13. Save trained artifacts

Saved files include model weights, preprocessing object, selected feature mask, and feature names. If Stage 2 was skipped, there is no GAN model to save.

In [ ]:
torch.save(capsnet.state_dict(), "improved_capsnet_cardio.pt")
torch.save(final_model.state_dict(), "gp dcnn_cardio.pt".replace(" ", ""))
joblib.dump(preprocessor, "cardio_preprocessor.joblib")
np.save("eaft_selected_feature_mask_cardio.npy", selected_mask)
np.save("cardio_selected_feature_names.npy", feature_names[selected_mask])

if gan is not None:
    torch.save(gan.state_dict(), "spe_acgan_inspired_cardio.pt")

print("Saved artifacts:")
for f in [
    "improved_capsnet_cardio.pt",
    "gp dcnn_cardio.pt".replace(" ", ""),
    "cardio_preprocessor.joblib",
    "eaft_selected_feature_mask_cardio.npy",
    "cardio_selected_feature_names.npy"
]:
    print("-", f)
if gan is not None:
    print("- spe_acgan_inspired_cardio.pt")

## 14. Interpretation and limitations

- Report **accuracy, balanced accuracy, precision, recall, F1, AUC, and the confusion matrix**; do not rely on accuracy alone.
- The CSV contains 70,000 rows originally, but the explicit sanity filters remove some records. Report the post-cleaning sample count.
- The dataset is observational and commonly used for educational prediction experiments. This model is not a clinical diagnostic device.
- A high score on this dataset does not establish generalization to other hospitals, populations, or prospective patients.
- Because the class distribution is already close to balanced, skipping GAN oversampling is intentional.
- This notebook uses adaptations of paper components; it does not establish exact reproduction of the paper's architecture or reported results.